In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

In [ ]:
EPOCHS = 32
BATCH_SIZE = 256
INPUT_FILE_NAME = '../datasets/frankenstein.txt'
WINDOW_LENGTH = 40
WINDOW_STEP = 3

LESS_EPOCHS = 15
MORE_BATCH = 512

In [ ]:
# Open the input file.
file = open(INPUT_FILE_NAME, 'r', encoding='utf-8')
text = file.read()
file.close()



In [ ]:
# Train Just Chapter 1

# # remove gutenberg boilerplate
# start_marker = "CHAPTER I."
# # end_marker   = "THE END."
# # Working with smaller dataset
# end_marker   = "CHAPTER II."

# start = text.find(start_marker)
# end   = text.find(end_marker)

# if start != -1 and end != -1:
#     text = text[start+len(start_marker):end].strip()   #.strip() removes leading and trailing whitespace from a string
# else:
#     text = text

In [ ]:
text

In [ ]:

# Make lowercase and remove newline and extra spaces.
text = text.lower()
text = text.replace('\n', ' ')
text = text.replace(' ', ' ')
text[:500]

In [ ]:
len(text)

In [ ]:
print(set(text))

In [ ]:
# Encode characters as indices.
unique_chars = list(set(text))
print(unique_chars)

In [ ]:

char_to_index = dict((ch, index) for index, ch in enumerate(unique_chars))
print(char_to_index)

In [ ]:

index_to_char = dict((index, ch) for index, ch in enumerate(unique_chars))
print(index_to_char)

In [ ]:
encoding_width = len(char_to_index)

In [ ]:
encoding_width

In [ ]:
# Create training examples.
fragments = []
targets = []

for i in range(0, len(text) - WINDOW_LENGTH, WINDOW_STEP):
    fragments.append(text[i: i + WINDOW_LENGTH])
    targets.append(text[i + WINDOW_LENGTH])

In [ ]:
print(fragments)

In [ ]:
len(fragments)

In [ ]:
# Convert to one-hot encoded training data.
X = np.zeros((len(fragments), WINDOW_LENGTH, encoding_width), dtype=np.float32)
y = np.zeros((len(fragments), encoding_width), dtype=np.float32)
for i, fragment in enumerate(fragments):
    for j, char in enumerate(fragment):
        X[i, j, char_to_index[char]] = 1
    target_char = targets[i]
    y[i, char_to_index[target_char]] = 1

In [ ]:
print(tf.config.list_physical_devices("GPU"))

In [ ]:
# Build and train model.
model = Sequential()
# model.add(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,input_shape=(None, encoding_width)))
# model.add(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,input_shape=(None, encoding_width)))
# model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
# Remove recurrent dropout for better performance on GPU
model.add(LSTM(128, return_sequences=True, dropout=0.2,input_shape=(None, encoding_width)))
model.add(LSTM(128, return_sequences=True, dropout=0.2,input_shape=(None, encoding_width)))
model.add(LSTM(128, dropout=0.2))
model.add(Dense(encoding_width, activation='softmax'))
model.compile(loss='categorical_crossentropy',optimizer='adam')
model.summary()

In [ ]:

history = model.fit(X, y, validation_split=0.05,batch_size=BATCH_SIZE,epochs=EPOCHS, verbose=2,shuffle=True)

In [ ]:
letters = 'the boy '
one_hots = []
for i, char in enumerate(letters):
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

new = np.array(one_hots).reshape(1, len(letters), encoding_width)

In [ ]:
new.shape

In [ ]:
y_predict = model.predict(new, verbose=1)

In [ ]:
y_predict

In [ ]:
np.argmax(y_predict)

In [ ]:
index_to_char[np.argmax(y_predict)]

In [ ]:
# Number of characters to generate
NUM_PREDICTIONS = 12

# Seed text
seed_text = "the boy who "
generated = seed_text

# Prepare one-hot input from the seed text
one_hots = []
for char in seed_text:
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

input_seq = np.array(one_hots).reshape(1, len(seed_text), encoding_width)

# Generate next NUM_PREDICTIONS characters
for i in range(NUM_PREDICTIONS):
    # Predict probabilities for next char
    y_predict = model.predict(input_seq, verbose=0)
    next_index = np.argmax(y_predict)
    next_char = index_to_char[next_index]

    print(f"Predicted letter {i+1}: {next_char}")
    generated += next_char

    # Update input sequence: append new char & drop oldest if longer than WINDOW_LENGTH
    next_one_hot = np.zeros((1, 1, encoding_width))
    next_one_hot[0, 0, next_index] = 1
    input_seq = np.concatenate([input_seq, next_one_hot], axis=1)

    if input_seq.shape[1] > WINDOW_LENGTH:
        input_seq = input_seq[:, 1:, :]   # keep only last WINDOW_LENGTH chars

print("\nGenerated text:\n", generated)

In [ ]:
## save model for reuse
model.save("../models/language_model_LSTM.keras")

In [ ]:
## load saved model
loaded_model = keras.models.load_model("../models/language_model_LSTM.keras")

In [ ]:
## predict using saved model
y_test = loaded_model(new)

In [ ]:
## convert index to character
index_to_char[np.argmax(y_test)]

In [ ]:
# Number of characters to generate
NUM_PREDICTIONS = 15
encoding_width=loaded_model.input_shape[-1] # use model's expected input width

# Seed text
seed_text = "the gulf "
generated = seed_text

# Prepare one-hot input from the seed text
one_hots = []
for char in seed_text:
    x = np.zeros(encoding_width)
    x[char_to_index[char]] = 1
    one_hots.append(x)

input_seq = np.array(one_hots).reshape(1, len(seed_text), encoding_width)

# Generate next NUM_PREDICTIONS characters
for i in range(NUM_PREDICTIONS):
    # Predict probabilities for next char
    y_predict = loaded_model.predict(input_seq, verbose=0)
    next_index = np.argmax(y_predict)
    next_char = index_to_char[next_index]

    print(f"Predicted letter {i+1}: {next_char}")
    generated += next_char

    # Update input sequence: append new char & drop oldest if longer than WINDOW_LENGTH
    next_one_hot = np.zeros((1, 1, encoding_width))
    next_one_hot[0, 0, next_index] = 1
    input_seq = np.concatenate([input_seq, next_one_hot], axis=1)

    if input_seq.shape[1] > WINDOW_LENGTH:
        input_seq = input_seq[:, 1:, :]   # keep only last WINDOW_LENGTH chars

print("\nGenerated text:\n", generated)